# Novelty signal table

Paste reviewer novelty signals into the next cell, run the one after it to eyeball
the layout, and the last one to get the LaTeX. Regenerates `tab:novelty_signals`
in `12_appendix.tex`.

Wrap the span that carries the novelty judgment in `**double asterisks**`; it
becomes `\sig{...}` (bold) in the output.

Reviewer prose is escaped literally, so **no LaTeX or math inside a quote
survives** -- strip any `$...$` before pasting. `CAPTION` is the exception: it is
passed through verbatim, so write real LaTeX there.

In [ ]:
# ---------------------------------------------------------------- paste here
# One (submission title, quote) per signal. Mark judgment spans with **asterisks**.

POSITIVE = [
    ("Image Quality Assessment for Embodied AI",
     "The paper identifies and clearly defines a **completely new, critical, and "
     "timely research problem**"),
    ("Quotient-Space Diffusion Models",
     "It is the **first work I have seen** that attempts to **constrain the "
     "diffusion process within the quotient space**"),
    ("Critical Confabulation: Can LLMs Hallucinate for Social Good?",
     "this paper is **so unlike most of the ICLR paper I've seen in recent years**, "
     "especially with its **use of historical archival data**…"),
]

NEGATIVE = [
    ("DFCA: Decentralized Federated Clustering Algorithm",
     "While the theoretical contribution is **not highly novel**, the paper provides "
     "a useful convergence discussion…"),
    ("NoLoRA: Nonlinear Low-Rank Adaptation for Parameter-Efficient Fine-Tuning",
     "The proposal is **extremely close to NEAT** (Zhong et al., 2025). Both methods "
     "replace LoRA's linear update with a nonlinear mapping…"),
    ("Consistent 3D Object Detection with Active LLM Reasoning",
     "The contribution is **primarily procedural integration of existing "
     "components** (YOLOE + CLIP + GPT-4V + SAM2), with **no novel learning "
     "formulation or analytical insight**…"),
]

CAPTION = (
    r"\textbf{Novelty signals extracted from ICLR 2026 reviews} by "
    r"\texttt{claude-opus-4-6}, three positive and three negative, each shown with "
    r"the submission it was taken from. \textbf{Bold} marks the span of the review "
    r"text that carries the novelty judgment: positive signals point at a problem, "
    r"formulation, or data source the reviewer has not seen before, while negative "
    r"signals point at closeness to a specific prior method or at assembly of "
    r"existing components."
)
LABEL = "tab:novelty_signals"


In [ ]:
# Escaping + the two renderers. Running this cell shows the preview.

import html
import re
from IPython.display import HTML, display

PAPER_W, QUOTE_W = 0.27, 0.69            # column widths, as fractions of \textwidth
HEADERS = ("ICLR 2026 submission", "Extracted reviewer novelty signal examples")
GROUPS = (("Positive novelty signals", "palDeepTeal", "#2f6b70"),
          ("Negative novelty signals", "palTerracotta", "#9c4a2f"))

_TEX = {"\\": r"\textbackslash{}", "{": r"\{", "}": r"\}", "&": r"\&", "%": r"\%",
        "$": r"\$", "#": r"\#", "_": r"\_", "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}"}
_UNI = {"—": "---", "–": "--", "‑": "-", "‘": "`", "’": "'",
        "“": "``", "”": "''", "…": r"\ldots{}",
        "→": r"$\rightarrow$"}


def tex(s):
    """Reviewer prose -> LaTeX. Escapes first, then maps the unicode we actually see."""
    s = " ".join(s.split())                      # collapse pasted line breaks
    s = "".join(_TEX.get(c, c) for c in s)
    s = "".join(_UNI.get(c, c) for c in s)
    odd = sorted({c for c in s if ord(c) > 127})
    if odd:
        print(f"! left as-is, check it compiles: {' '.join(odd)}")
    return s


def _spans(s, wrap):
    """`**span**` -> wrap(span). Runs after escaping, so what it adds is left alone."""
    if s.count("**") % 2:
        print(f"! odd number of ** markers: {s[:70]}...")
    return re.sub(r"\*\*(.+?)\*\*", lambda m: wrap(m.group(1)), s, flags=re.S)


def _rows():
    """(heading, tex colour, css colour, rows) for each non-empty group."""
    items = (POSITIVE, NEGATIVE)
    return [(h, tc, cc, r) for (h, tc, cc), r in zip(GROUPS, items) if r]


def preview():
    """Rough HTML render: check length and wrapping before compiling.

    Runs every field through `tex()` so escaping warnings surface here rather than
    at LaTeX time. Greyed counts are a fitting aid, not part of the table.
    """
    body = []
    for heading, _, css, rows in _rows():
        body.append(f'<tr><td colspan="2" style="padding:10px 5px 2px;color:{css};'
                    f'font-weight:bold">{html.escape(heading)}</td></tr>')
        for paper, quote in rows:
            tex(paper), tex(quote)
            marked = _spans(html.escape(quote), lambda t: f"<b>{t}</b>")
            body.append(
                '<tr><td style="padding:4px 5px;vertical-align:top">'
                f'<i>{html.escape(paper)}</i>'
                f'<div style="opacity:.4;font-size:11px">{len(quote)} chars</div></td>'
                '<td style="padding:4px 5px;vertical-align:top;text-align:justify">'
                f'“{marked}”</td></tr>')
    display(HTML(
        '<div style="font-family:Georgia,\'Times New Roman\',serif;font-size:13px">'
        '<table style="width:100%;border-collapse:collapse;'
        'border-top:1.5px solid currentColor;border-bottom:1.5px solid currentColor">'
        f'<colgroup><col style="width:{PAPER_W / (PAPER_W + QUOTE_W):.1%}"><col>'
        '</colgroup>'
        '<tr style="border-bottom:.8px solid currentColor">'
        + "".join(f'<th style="padding:5px;text-align:left">{h}</th>' for h in HEADERS)
        + '</tr>' + "".join(body) +
        f'</table><div style="padding:8px 0;font-size:12px">Caption: {CAPTION}</div>'
        '</div>'))


def latex_table(caption=None, label=None):
    caption, label = caption or CAPTION, label or LABEL
    lines = [
        r"\begin{table*}[t]",
        r"\centering",
        r"\footnotesize",
        r"\setlength{\tabcolsep}{5pt}",
        r"\renewcommand{\arraystretch}{1.2}",
        r"% \sig marks the span of text that carries the novelty judgment.",
        # Defined inside the float, so it is local to this group and a second such
        # table does not clash with it.
        r"\newcommand{\sig}[1]{\textbf{#1}}",
        rf"\begin{{tabular}}{{@{{}}p{{{PAPER_W}\textwidth}}p{{{QUOTE_W}\textwidth}}@{{}}}}",
        r"\toprule",
        " & ".join(rf"\textbf{{{h}}}" for h in HEADERS) + r" \\",
    ]
    for heading, colour, _, rows in _rows():
        lines.append(r"\midrule")
        lines.append(rf"\multicolumn{{2}}{{@{{}}l}}{{\textcolor{{{colour}}}"
                     rf"{{\textbf{{{heading}}}}}}} \\")
        for j, (paper, quote) in enumerate(rows):
            lines.append(rf"\addlinespace[{2 if j == 0 else 4}pt]")
            lines.append(rf"\textit{{{tex(paper)}}} &")
            lines.append(rf"``{_spans(tex(quote), lambda t: rf'\sig{{{t}}}')}'' \\")
    lines += [
        r"\bottomrule",
        r"\end{tabular}",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\end{table*}",
    ]
    return "\n".join(lines)


preview()


In [ ]:
print(latex_table())


The preview approximates; only the compiled PDF is authoritative about
line breaks. `palDeepTeal` / `palTerracotta` come from the paper preamble
(`acl_latex.tex`), and the column widths deliberately sum to 0.96 rather than 1.0,
matching the table as hand-written.